In [1]:
import ctypes
import numpy as np
from pathlib import Path

lib = ctypes.CDLL("./linear_model_lib.so")

pf = ctypes.POINTER(ctypes.c_float)
pi = ctypes.POINTER(ctypes.c_int)

lib.fit.argtypes = [pf, pi, ctypes.c_int, ctypes.c_int,
                    ctypes.c_int, ctypes.c_float, pf, pf]
lib.fit.restype = None

lib.predict.argtypes = [pf, ctypes.c_int, ctypes.c_int, pf, pf, pi]
lib.predict.restype = None

In [2]:
def lire_X(chemin):
    with open(chemin, "rb") as f:
        n = int(np.fromfile(f, dtype=np.int32, count=1)[0])
        d = int(np.fromfile(f, dtype=np.int32, count=1)[0])
        X = np.fromfile(f, dtype=np.float32, count=n * d)
    return np.ascontiguousarray(X.reshape(n, d)), n, d

def lire_y(chemin):
    with open(chemin, "rb") as f:
        n = int(np.fromfile(f, dtype=np.int32, count=1)[0])
        y = np.fromfile(f, dtype=np.int32, count=n)
    return np.ascontiguousarray(y), n

racine = Path.cwd()
while racine != racine.parent and not (racine / "preprocessing").exists():
    racine = racine.parent
base = racine / "datasets" / "transformed" / "nb" / "normalisee"

X_train, n_train, d = lire_X(base / "X_train.f32bin")
y_train, _ = lire_y(base / "y_train.i32bin")
X_test, n_test, _ = lire_X(base / "X_test.f32bin")
y_test, _ = lire_y(base / "y_test.i32bin")
print(n_train, n_test, d)

1200 300 16384


In [3]:
epochs, lr = 30, 0.1
K_CLASSES = 3

W = np.zeros(K_CLASSES * d, dtype=np.float32)
b = np.zeros(K_CLASSES, dtype=np.float32)

lib.fit(
    X_train.ctypes.data_as(pf), y_train.ctypes.data_as(pi),
    n_train, d, epochs, ctypes.c_float(lr),
    W.ctypes.data_as(pf), b.ctypes.data_as(pf),
)

predictions = np.zeros(n_test, dtype=np.int32)
lib.predict(
    X_test.ctypes.data_as(pf), n_test, d,
    W.ctypes.data_as(pf), b.ctypes.data_as(pf),
    predictions.ctypes.data_as(pi),
)

acc_test = (predictions == y_test).mean()
print("acc test (lib lineaire) :", round(acc_test, 3))

acc test (lib lineaire) : 0.42
